# Modele 1 - Content-Based : Similarite Cosinus sur Embeddings

Objectif : recommander 5 articles par utilisateur en se basant sur la proximite semantique (cosinus) entre les articles lus et les articles candidats.


## 1. Chargement des donnees


In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import json
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix
import joblib
import matplotlib.pyplot as plt

DATA_RAW = "../data/raw/"
DATA_PROC = "../data/processed/"
OUT = "../out/"
CB_DIR = os.path.join(OUT, "models", "content_based")
FIG = os.path.join(OUT, "figures")
os.makedirs(OUT, exist_ok=True)
os.makedirs(CB_DIR, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

In [2]:
# Clicks enrichis (depuis l'EDA)
clicks = pd.read_parquet(os.path.join(DATA_PROC, "clicks_enriched.parquet"))
interactions = pd.read_parquet(os.path.join(DATA_PROC, "interactions.parquet"))

# Metadata articles
articles = pd.read_csv(os.path.join(DATA_RAW, "articles_metadata.csv"))

# Embeddings
with open(os.path.join(DATA_RAW, "articles_embeddings.pickle"), "rb") as f:
    embeddings_raw = pickle.load(f)

print(f"Clicks : {len(clicks):,} lignes")
print(f"Interactions uniques : {len(interactions):,}")
print(f"Articles metadata : {len(articles):,}")
print(f"Embeddings : {embeddings_raw.shape}")

/tmp/ipykernel_7967/322960300.py:10: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  embeddings_raw = pickle.load(f)


Clicks : 2,988,181 lignes
Interactions uniques : 2,950,710
Articles metadata : 364,047
Embeddings : (364047, 250)


## 2. Preparation des embeddings


Les embeddings sont un ndarray (364047, 250). L'index correspond directement a l'article_id (article_id 0 = ligne 0, etc.).


In [3]:
# Verification de la correspondance index / article_id
print(f"Range article_id dans metadata : {articles['article_id'].min()} -> {articles['article_id'].max()}")
print(f"Nombre de lignes embeddings : {len(embeddings_raw)}")
print(f"Correspondance directe : article_id == index ligne")

# Articles effectivement cliques
clicked_article_ids = set(clicks["click_article_id"].unique())
print(f"\nArticles cliques : {len(clicked_article_ids):,}")
print(f"Tous couverts par embeddings : {all(aid < len(embeddings_raw) for aid in clicked_article_ids)}")

Range article_id dans metadata : 0 -> 364046
Nombre de lignes embeddings : 364047
Correspondance directe : article_id == index ligne

Articles cliques : 46,033
Tous couverts par embeddings : True


### 2.1 Normalisation L2

Pour que la similarite cosinus revienne a un simple produit scalaire (gain de performance), on normalise les vecteurs.


In [4]:
# Normalisation L2
embeddings_norm = normalize(embeddings_raw, norm="l2", axis=1)
print(f"Shape : {embeddings_norm.shape}")
print(f"Norme L2 du premier vecteur : {np.linalg.norm(embeddings_norm[0]):.4f}")

Shape : (364047, 250)
Norme L2 du premier vecteur : 1.0000


### 2.2 Reduction de dimension (ACP)

Les embeddings font 250 dimensions. L'EDA a montre qu'il faut environ 40 composantes pour capter 90% de la variance. On teste plusieurs niveaux pour trouver le bon compromis qualite/performance, utile pour le deploiement Lambda.


In [5]:
# ACP sur les embeddings complets
n_components_list = [20, 30, 40, 50, 80]
pca_results = {}

for n in n_components_list:
    pca = PCA(n_components=n)
    emb_reduced = pca.fit_transform(embeddings_raw)
    var_explained = pca.explained_variance_ratio_.sum()
    pca_results[n] = {
        "variance": var_explained,
        "pca": pca,
        "embeddings": emb_reduced
    }
    print(f"n={n:3d} -> variance expliquee : {var_explained:.3%} | taille memoire : {emb_reduced.nbytes / 1e6:.1f} Mo")

print(f"\nOriginal : {embeddings_raw.nbytes / 1e6:.1f} Mo")

n= 20 -> variance expliquee : 70.669% | taille memoire : 29.1 Mo
n= 30 -> variance expliquee : 83.024% | taille memoire : 43.7 Mo
n= 40 -> variance expliquee : 90.288% | taille memoire : 58.2 Mo
n= 50 -> variance expliquee : 94.530% | taille memoire : 72.8 Mo
n= 80 -> variance expliquee : 98.211% | taille memoire : 116.5 Mo

Original : 364.0 Mo


In [6]:
# Choix : on garde 50 composantes
N_COMPONENTS = 50
pca_model = pca_results[N_COMPONENTS]["pca"]
embeddings_reduced = pca_results[N_COMPONENTS]["embeddings"]
embeddings_reduced_norm = normalize(embeddings_reduced, norm="l2", axis=1)

print(f"Embeddings reduits : {embeddings_reduced_norm.shape}")
print(f"Variance conservee : {pca_results[N_COMPONENTS]['variance']:.1%}")
print(f"Reduction taille : {embeddings_raw.nbytes / 1e6:.1f} Mo -> {embeddings_reduced_norm.nbytes / 1e6:.1f} Mo")

Embeddings reduits : (364047, 50)
Variance conservee : 94.5%
Reduction taille : 364.0 Mo -> 72.8 Mo


## 3. Construction du profil utilisateur

Pour chaque utilisateur, son profil est le vecteur moyen des embeddings des articles qu'il a lus. C'est l'approche la plus classique en content-based filtering.


In [7]:
# Articles lus par chaque utilisateur
user_articles = clicks.groupby("user_id")["click_article_id"].apply(list).to_dict()

print(f"Utilisateurs : {len(user_articles):,}")
print(f"Exemple user 0 : articles lus = {user_articles[0]}")

Utilisateurs : 322,897
Exemple user 0 : articles lus = [157541, 68866, 96755, 313996, 160158, 233470, 87224, 87205]


In [8]:
def build_user_profile(user_id, user_articles, embeddings):
    """Profil utilisateur = moyenne des embeddings des articles lus."""
    article_ids = user_articles.get(user_id, [])
    if not article_ids:
        return None
    # Filtrer les articles valides
    valid_ids = [aid for aid in article_ids if aid < len(embeddings)]
    if not valid_ids:
        return None
    profile = embeddings[valid_ids].mean(axis=0)
    # Normaliser le profil
    norm = np.linalg.norm(profile)
    if norm > 0:
        profile = profile / norm
    return profile

# Test sur un utilisateur
profile_0 = build_user_profile(0, user_articles, embeddings_reduced)
print(f"Profil user 0 : shape={profile_0.shape}, norme={np.linalg.norm(profile_0):.4f}")

Profil user 0 : shape=(50,), norme=1.0000


## 4. Fonction de recommandation


In [9]:
def recommend_content_based(user_id, user_articles, embeddings, top_n=5):
    """Recommande top_n articles par similarite cosinus."""
    profile = build_user_profile(user_id, user_articles, embeddings)
    if profile is None:
        return []
    
    # Similarite entre le profil et tous les articles
    scores = embeddings.dot(profile)
    
    # Exclure les articles deja lus
    read_articles = set(user_articles.get(user_id, []))
    
    # Ranking
    ranked_indices = np.argsort(scores)[::-1]
    
    recommendations = []
    for idx in ranked_indices:
        if idx not in read_articles and len(recommendations) < top_n:
            recommendations.append({
                "article_id": int(idx),
                "score": float(scores[idx])
            })
        if len(recommendations) >= top_n:
            break
    
    return recommendations

In [10]:
# Test sur quelques utilisateurs
test_users = [0, 1, 5, 100, 1000]

for uid in test_users:
    recs = recommend_content_based(uid, user_articles, embeddings_reduced_norm, top_n=5)
    articles_lus = user_articles.get(uid, [])
    print(f"User {uid} | Articles lus : {len(articles_lus)} | Recommandations :")
    for r in recs:
        # Enrichir avec metadata
        meta = articles[articles["article_id"] == r["article_id"]]
        cat = meta["category_id"].values[0] if len(meta) > 0 else "?"
        wc = meta["words_count"].values[0] if len(meta) > 0 else "?"
        print(f"  article_id={r['article_id']:>6d} | score={r['score']:.4f} | cat={cat} | mots={wc}")
    print()

User 0 | Articles lus : 8 | Recommandations :
  article_id=161133 | score=0.8672 | cat=281 | mots=184
  article_id=107637 | score=0.8647 | cat=228 | mots=217
  article_id=162235 | score=0.8623 | cat=281 | mots=222
  article_id=161789 | score=0.8593 | cat=281 | mots=181
  article_id=159495 | score=0.8581 | cat=281 | mots=261

User 1 | Articles lus : 12 | Recommandations :
  article_id=238038 | score=0.7904 | cat=376 | mots=223
  article_id=161547 | score=0.7862 | cat=281 | mots=177
  article_id=158999 | score=0.7849 | cat=281 | mots=280
  article_id=160924 | score=0.7828 | cat=281 | mots=238
  article_id=154444 | score=0.7785 | cat=281 | mots=176

User 5 | Articles lus : 87 | Recommandations :
  article_id= 42943 | score=0.8768 | cat=67 | mots=217
  article_id=104516 | score=0.8608 | cat=228 | mots=195
  article_id=158992 | score=0.8440 | cat=281 | mots=155
  article_id=283276 | score=0.8417 | cat=412 | mots=205
  article_id=346110 | score=0.8350 | cat=440 | mots=236

User 100 | Article

## 5. Analyse qualitative des recommandations

On verifie que les articles recommandes sont coherents avec l'historique de l'utilisateur (meme categorie, memes themes).


In [11]:
def analyze_recommendations(user_id, user_articles, embeddings, articles_meta, top_n=5):
    """Analyse la coherence thematique des recommandations."""
    recs = recommend_content_based(user_id, user_articles, embeddings, top_n=top_n)
    read_ids = user_articles.get(user_id, [])
    
    # Categories lues
    read_cats = articles_meta[articles_meta["article_id"].isin(read_ids)]["category_id"].value_counts()
    
    # Categories recommandees
    rec_ids = [r["article_id"] for r in recs]
    rec_cats = articles_meta[articles_meta["article_id"].isin(rec_ids)]["category_id"].value_counts()
    
    return {
        "user_id": user_id,
        "nb_read": len(read_ids),
        "read_categories": read_cats.to_dict(),
        "rec_categories": rec_cats.to_dict(),
        "overlap_categories": set(read_cats.index) & set(rec_cats.index)
    }

# Analyser un echantillon
sample_users = np.random.choice(list(user_articles.keys()), size=100, replace=False)
overlaps = []

for uid in sample_users:
    result = analyze_recommendations(uid, user_articles, embeddings_reduced_norm, articles)
    if result["nb_read"] > 0 and len(result["rec_categories"]) > 0:
        # Proportion de categories recommandees qui matchent les categories lues
        read_set = set(result["read_categories"].keys())
        rec_set = set(result["rec_categories"].keys())
        overlap_ratio = len(read_set & rec_set) / len(rec_set) if rec_set else 0
        overlaps.append(overlap_ratio)

print(f"Coherence categorielle sur {len(overlaps)} users :")
print(f"  Moyenne : {np.mean(overlaps):.1%}")
print(f"  Mediane : {np.median(overlaps):.1%}")
print(f"  Users avec au moins 1 categorie commune : {sum(1 for o in overlaps if o > 0) / len(overlaps):.1%}")

Coherence categorielle sur 100 users :
  Moyenne : 69.7%
  Mediane : 66.7%
  Users avec au moins 1 categorie commune : 96.0%


## 6. Benchmark de performance


In [12]:
# Temps de recommandation pour 1 user
import time

uid_test = 100
times = []
for _ in range(100):
    t0 = time.time()
    recommend_content_based(uid_test, user_articles, embeddings_reduced_norm, top_n=5)
    times.append(time.time() - t0)

print(f"Temps moyen par recommandation : {np.mean(times)*1000:.1f} ms")
print(f"Temps median : {np.median(times)*1000:.1f} ms")
print(f"Temps max : {np.max(times)*1000:.1f} ms")
print(f"Compatible Lambda (<1s) : {'Oui' if np.mean(times) < 1 else 'Non'}")

Temps moyen par recommandation : 16.1 ms
Temps median : 15.4 ms
Temps max : 24.5 ms
Compatible Lambda (<1s) : Oui


In [13]:
# Comparaison embeddings 250d vs reduits
times_full = []
embeddings_full_norm = normalize(embeddings_raw, norm="l2", axis=1)

for _ in range(50):
    t0 = time.time()
    recommend_content_based(uid_test, user_articles, embeddings_full_norm, top_n=5)
    times_full.append(time.time() - t0)

times_reduced = []
for _ in range(50):
    t0 = time.time()
    recommend_content_based(uid_test, user_articles, embeddings_reduced_norm, top_n=5)
    times_reduced.append(time.time() - t0)

print(f"Embeddings 250d : {np.mean(times_full)*1000:.1f} ms | taille : {embeddings_full_norm.nbytes / 1e6:.1f} Mo")
print(f"Embeddings {N_COMPONENTS}d  : {np.mean(times_reduced)*1000:.1f} ms | taille : {embeddings_reduced_norm.nbytes / 1e6:.1f} Mo")
print(f"Speedup : x{np.mean(times_full)/np.mean(times_reduced):.1f}")

Embeddings 250d : 24.2 ms | taille : 364.0 Mo
Embeddings 50d  : 15.6 ms | taille : 72.8 Mo
Speedup : x1.6


## 7. Baseline : recommandation par popularite

Pour mesurer la valeur ajoutee du content-based, on compare avec une baseline naive : recommander les 5 articles les plus populaires (non lus par le user).


In [14]:
# Articles les plus populaires
article_popularity = clicks.groupby("click_article_id").size().sort_values(ascending=False)
popular_articles = article_popularity.index.tolist()

def recommend_popularity(user_id, user_articles, popular_articles, top_n=5):
    """Baseline : articles les plus populaires non lus."""
    read_articles = set(user_articles.get(user_id, []))
    recs = []
    for aid in popular_articles:
        if aid not in read_articles:
            recs.append({"article_id": int(aid), "score": float(article_popularity[aid])})
        if len(recs) >= top_n:
            break
    return recs

# Comparaison sur quelques users
for uid in [0, 1, 100]:
    recs_cb = recommend_content_based(uid, user_articles, embeddings_reduced_norm, top_n=5)
    recs_pop = recommend_popularity(uid, user_articles, popular_articles, top_n=5)
    
    cb_ids = [r["article_id"] for r in recs_cb]
    pop_ids = [r["article_id"] for r in recs_pop]
    
    print(f"User {uid}:")
    print(f"  Content-Based : {cb_ids}")
    print(f"  Popularite    : {pop_ids}")
    print(f"  Overlap       : {len(set(cb_ids) & set(pop_ids))} / 5")
    print()

User 0:
  Content-Based : [161133, 107637, 162235, 161789, 159495]
  Popularite    : [160974, 272143, 336221, 234698, 123909]
  Overlap       : 0 / 5

User 1:
  Content-Based : [238038, 161547, 158999, 160924, 154444]
  Popularite    : [160974, 272143, 336221, 234698, 123909]
  Overlap       : 0 / 5

User 100:
  Content-Based : [331228, 331762, 332420, 330477, 332449]
  Popularite    : [160974, 272143, 336221, 234698, 123909]
  Overlap       : 0 / 5



## 8. Fonction articles similaires

Utile pour le cold start : si un nouvel utilisateur consulte un article, on peut immediatement lui recommander des articles proches.


In [15]:
def get_similar_articles(article_id, embeddings, top_n=5):
    """Retourne les top_n articles les plus similaires a article_id."""
    if article_id >= len(embeddings):
        return []
    
    query = embeddings[article_id]
    scores = embeddings.dot(query)
    scores[article_id] = -1  # Exclure l'article lui-meme
    
    top_indices = np.argsort(scores)[::-1][:top_n]
    
    return [{"article_id": int(idx), "score": float(scores[idx])} for idx in top_indices]

# Test : articles similaires au top article
top_article = article_popularity.index[0]
similar = get_similar_articles(top_article, embeddings_reduced_norm, top_n=10)
print(f"Articles similaires a l'article {top_article} (le plus populaire) :")
for s in similar:
    meta = articles[articles["article_id"] == s["article_id"]]
    cat = meta["category_id"].values[0] if len(meta) > 0 else "?"
    pop = article_popularity.get(s["article_id"], 0)
    print(f"  article_id={s['article_id']:>6d} | score={s['score']:.4f} | cat={cat} | clics={pop}")

Articles similaires a l'article 160974 (le plus populaire) :
  article_id=156560 | score=0.8948 | cat=281 | clics=14417
  article_id=150616 | score=0.8819 | cat=281 | clics=0
  article_id=161100 | score=0.8666 | cat=281 | clics=1135
  article_id=158828 | score=0.8662 | cat=281 | clics=0
  article_id=162338 | score=0.8634 | cat=281 | clics=1223
  article_id=159764 | score=0.8610 | cat=281 | clics=5
  article_id=159034 | score=0.8608 | cat=281 | clics=279
  article_id=158983 | score=0.8582 | cat=281 | clics=0
  article_id=162172 | score=0.8580 | cat=281 | clics=4
  article_id=160953 | score=0.8570 | cat=281 | clics=0


## 9. Sauvegarde des artefacts pour le deploiement


In [16]:
# Sauvegarder les elements necessaires au deploiement
# 1. Modele ACP pour transformer de nouveaux embeddings
joblib.dump(pca_model, os.path.join(CB_DIR, "pca_model.joblib"))

# 2. Embeddings reduits et normalises (ce qui est charge par la Lambda)
np.save(os.path.join(CB_DIR, "embeddings_reduced.npy"), embeddings_reduced_norm)

# 3. Historique utilisateur (articles lus par user)
with open(os.path.join(CB_DIR, "user_articles.json"), "w") as f:
    json.dump({str(k): v for k, v in user_articles.items()}, f)

# 4. Popularite articles (fallback)
article_popularity.to_frame("nb_clicks").to_parquet(os.path.join(CB_DIR, "article_popularity.parquet"))

# 5. Metadata articles
cfg_dir = os.path.join(OUT, "models", "config")
os.makedirs(cfg_dir, exist_ok=True)
articles.to_parquet(os.path.join(cfg_dir, "articles_metadata.parquet"), index=False)

print("Artefacts sauvegardes :")
for root, dirs, files in os.walk(os.path.join(OUT, "models")):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1e6
        rel = os.path.relpath(fpath, OUT)
        print(f"  {rel} : {size:.1f} Mo")

Artefacts sauvegardes :
  models/config/articles_metadata.parquet : 4.9 Mo
  models/config/model_config.json : 0.0 Mo
  models/collaborative/test_ground_truth.json : 5.6 Mo
  models/collaborative/user_item_matrix.npz : 5.5 Mo
  models/collaborative/als_model.npz : 73.8 Mo
  models/collaborative/article_map_inv.json : 0.8 Mo
  models/collaborative/als_model.joblib : 73.8 Mo
  models/collaborative/article_map.json : 0.8 Mo
  models/collaborative/user_map.json : 5.6 Mo
  models/content_based/user_articles.json : 27.1 Mo
  models/content_based/article_popularity.parquet : 0.3 Mo
  models/content_based/cold_start_ranking.parquet : 8.3 Mo
  models/content_based/pca_model.joblib : 0.1 Mo
  models/content_based/embeddings_reduced.npy : 72.8 Mo


## Cold start : score recence + popularite

Pour les nouveaux utilisateurs sans historique, on combine la recence de publication et la popularite globale de chaque article.


In [17]:
# Score recence + popularite pour le cold start
articles_scored = articles.copy()
articles_scored["recency_score"] = (articles_scored["created_at_ts"] - articles_scored["created_at_ts"].min()) / \
    (articles_scored["created_at_ts"].max() - articles_scored["created_at_ts"].min())

pop_counts = clicks.groupby("click_article_id").size().reset_index(name="nb_clicks")
articles_scored = articles_scored.merge(pop_counts, left_on="article_id", right_on="click_article_id", how="left")
articles_scored["nb_clicks"] = articles_scored["nb_clicks"].fillna(0)
articles_scored["popularity_score"] = articles_scored["nb_clicks"] / articles_scored["nb_clicks"].max()

# Score combine : 0.4 * recence + 0.6 * popularite
articles_scored["cold_start_score"] = 0.4 * articles_scored["recency_score"] + 0.6 * articles_scored["popularity_score"]
articles_scored = articles_scored.sort_values("cold_start_score", ascending=False)

articles_scored[["article_id", "cold_start_score", "recency_score", "popularity_score"]].to_parquet(
    os.path.join(CB_DIR, "cold_start_ranking.parquet"), index=False
)

print("Top 10 articles cold start :")
print(articles_scored[["article_id", "category_id", "cold_start_score", "recency_score", "popularity_score"]].head(10).to_string(index=False))

Top 10 articles cold start :
 article_id  category_id  cold_start_score  recency_score  popularity_score
     160974          281          0.984479       0.961198          1.000000
     272143          399          0.851193       0.961334          0.777766
     336221          437          0.769813       0.963135          0.640932
     234698          375          0.764144       0.963150          0.631473
     123909          250          0.757601       0.961989          0.621342
     336223          437          0.737567       0.962976          0.587295
      96210          209          0.733354       0.963648          0.579824
     162655          281          0.724111       0.961300          0.565985
     183176          301          0.712738       0.963462          0.545589
     168623          297          0.699560       0.961837          0.524709
